# Bloc 03 - Conversion rate challenge

## 03 - Best model

### Import libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report

### Lecture du dataset

In [3]:
data = pd.read_csv('../data/conversion_data_train.csv')
print('Set with labels (our train+test) :', data.shape)

Set with labels (our train+test) : (284580, 6)


## Filtrage des âges

In [4]:
# on ne garde que les donnees dont la frequence d'âge est > 50
age_counts = data["age"].value_counts()
frequent_ages = age_counts[age_counts >= 50].index
data = data[data["age"].isin(frequent_ages)]

In [5]:
data.shape

(284486, 6)

## Entraînement d'un modèle

### Choix des variables

In [6]:
target_variable = 'converted'
X = data.drop(columns=target_variable)
y = data[target_variable]

numeric_columns = X.select_dtypes(exclude="object").columns.to_list()
cat_columns = X.select_dtypes(include="object").columns.to_list()

features_list = numeric_columns + cat_columns

print(numeric_columns, cat_columns)

print(features_list)

['age', 'new_user', 'total_pages_visited'] ['country', 'source']
['age', 'new_user', 'total_pages_visited', 'country', 'source']


### Train set & Test set

In [7]:
print("Dividing into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing

In [8]:
print("Encoding categorical features and standardizing numerical features...")

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

cat_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(drop="first"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", cat_transformer, cat_columns)
    ]
)

Encoding categorical features and standardizing numerical features...


### Application du preprocessing sur train

In [9]:
X_train = preprocessor.fit_transform(X_train)
print("...Done")
print(X_train[0:5,:])

...Done
[[-1.52260584  0.67696416 -0.56072291  0.          0.          1.
   0.          0.        ]
 [-0.67341753 -1.47718308  1.83509434  0.          0.          1.
   0.          1.        ]
 [ 1.14627171  0.67696416  1.23614003  0.          0.          1.
   0.          0.        ]
 [ 0.53970863 -1.47718308  0.93666287  0.          0.          0.
   1.          0.        ]
 [ 0.78233386 -1.47718308 -1.15967722  0.          0.          0.
   0.          1.        ]]


### Application du preprocessing sur test

In [10]:
X_test = preprocessor.transform(X_test)
print("...Done")
print(X_test[0:5,:])

...Done
[[ 0.2970834   0.67696416 -0.26124575  0.          0.          0.
   0.          1.        ]
 [-1.15866799  0.67696416 -0.86020006  1.          0.          0.
   1.          0.        ]
 [-0.67341753 -1.47718308  1.53561719  0.          0.          0.
   0.          1.        ]
 [-0.18816707  0.67696416 -0.26124575  0.          0.          1.
   0.          0.        ]
 [ 1.02495909  0.67696416  0.03823141  0.          0.          1.
   0.          1.        ]]


### Modèle

In [11]:
# Train model
print("Train model...")
classifier = LogisticRegression(class_weight={1:1.5}, C=0.2)
classifier.fit(X_train, y_train)
print("...Done.")

Train model...
...Done.


### Prédiction sur le training set

In [12]:
# Predictions on training set
print("Predictions on training set...")
y_train_pred = classifier.predict(X_train)
print("...Done.")
print(y_train_pred)
print()

Predictions on training set...
...Done.
[0 0 0 ... 0 0 0]



### Prédiction sur le test set

In [13]:
# Predictions on test set
print("Predictions on test set...")
y_test_pred = classifier.predict(X_test)
print("...Done.")
print(y_test_pred)
print()

Predictions on test set...
...Done.
[0 0 0 ... 0 0 0]



### Evaluation des performances

In [14]:
print("f1-score on train set : ", f1_score(y_train, y_train_pred))
print("f1-score on test set : ", f1_score(y_test, y_test_pred))

f1-score on train set :  0.7718269574544154
f1-score on test set :  0.7671154397020911


In [15]:
# Print classification report:
print('Classification report on train set:')
print(classification_report(y_train, y_train_pred))

print('Classification report on test set:')
print(classification_report(y_test, y_test_pred))

Classification report on train set:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    220246
           1       0.81      0.74      0.77      7342

    accuracy                           0.99    227588
   macro avg       0.90      0.86      0.88    227588
weighted avg       0.99      0.99      0.99    227588

Classification report on test set:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     55062
           1       0.81      0.73      0.77      1836

    accuracy                           0.99     56898
   macro avg       0.90      0.86      0.88     56898
weighted avg       0.99      0.99      0.99     56898



La classe 0 est toujours bien reconnu par le modèle.  

Pour la classe 1,
precision de 81%: le modèle a raison pour 81% du temps  
81% des utilisateurs vont réellement souscrire à la newsletter.

recall de 73%: le modèle détecte 73% de vrais positifs --> il rate 27% de cas positifs  
Le modèle identifie 73% des utilisateurs qui souscrivent à la newsletter.

In [16]:
# You can also check more performance metrics to better understand what your model is doing
print("Confusion matrix on train set : ")
print(confusion_matrix(y_train, y_train_pred))
print()
print("Confusion matrix on test set : ")
print(confusion_matrix(y_test, y_test_pred))
print()

Confusion matrix on train set : 
[[219000   1246]
 [  1945   5397]]

Confusion matrix on test set : 
[[54746   316]
 [  497  1339]]



Sur le Train set, la classe 0 a été correctement prédite 219 000 fois.  
La classe 1 a été prédite 5 397 fois.  

La classe 1 n'a pas été correctement prédite 1 246 fois.  
La classe 0 n'a pas été correctement prédite 1 945 fois.

### Nouvelles prédictions sur conversion_data_test.csv

In [17]:
# Concatenate our train and test set to train your best classifier on all data with labels
X = np.append(X_train,X_test,axis=0)
y = np.append(y_train,y_test)

classifier.fit(X,y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.2
,fit_intercept,True
,intercept_scaling,1
,class_weight,{1: 1.5}
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [19]:
# Read data without labels
data_without_labels = pd.read_csv('../data/conversion_data_test.csv')
print('Prediction set (without labels) :', data_without_labels.shape)

# Warning : check consistency of features_list (must be the same than the features 
# used by your best classifier)
X_without_labels = data_without_labels.loc[:, features_list]

print(X_without_labels.head())

Prediction set (without labels) : (31620, 5)
   age  new_user  total_pages_visited country  source
0   28         0                   16      UK     Seo
1   22         1                    5      UK  Direct
2   32         1                    1   China     Seo
3   32         1                    6      US     Ads
4   25         0                    3   China     Seo


In [20]:
# WARNING : PUT HERE THE SAME PREPROCESSING AS FOR YOUR TEST SET
# CHECK YOU ARE USING X_without_labels
print("Encoding categorical features and standardizing numerical features...")

X_without_labels = preprocessor.transform(X_without_labels)
print("...Done")
print(X_without_labels[0:5,:])

Encoding categorical features and standardizing numerical features...
...Done
[[-0.30947968 -1.47718308  3.33248012  0.          1.          0.
   0.          1.        ]
 [-1.03735538  0.67696416  0.03823141  0.          1.          0.
   1.          0.        ]
 [ 0.17577078  0.67696416 -1.15967722  0.          0.          0.
   0.          1.        ]
 [ 0.17577078  0.67696416  0.33770856  0.          0.          1.
   0.          0.        ]
 [-0.67341753 -1.47718308 -0.56072291  0.          0.          0.
   0.          1.        ]]


In [22]:
# Make predictions and dump to file
# WARNING : MAKE SURE THE FILE IS A CSV WITH ONE COLUMN NAMED 'converted' AND NO INDEX !
# WARNING : FILE NAME MUST HAVE FORMAT 'conversion_data_test_predictions_[name].csv'
# where [name] is the name of your team/model separated by a '-'
# For example : [name] = AURELIE-model1
data = {
    'converted': classifier.predict(X_without_labels)
}

y_predictions = pd.DataFrame(columns=['converted'],data=data)
current_dateTime  = datetime.now()
print(current_dateTime )
y_predictions.to_csv(f'../data/conversion_data_test_predictions_VICTOR-{current_dateTime .strftime("%Y-%m-%d-%H-%M-%S")}.csv', index=False)

2026-03-06 11:32:00.174870


## Conclusion

J'ai introduit 2 nouveaux paramètres au modèle LogisticRegression.  
class_weight={1:1.5} : les erreurs sur la classe 1 comptent 1.5 fois plus lors de l'entraînement  
donc le modèle tentera de prédire plus de 1  

C=0.2 : utilisation d'une régularisation modérée pour éviter le surapprentissage

### Recommendations

La Chine étant le 2e marché en terme de visiteurs, créer une version entièrement traduite en chinois.  
De même pour l'Allemagne, traduire la newsletter en allemand.  

Améliorer le site afin d'amener plus de trafic, et potentiellement plus de souscriptions.